In [94]:
import os
import sys
import random
from typing import Dict, Optional
import logging

from rdkit import Chem
from rdkit.Chem import Draw

curr_file_dir = os.getcwd()
parent_dir = os.path.dirname(curr_file_dir)
sys.path.append(os.path.join(parent_dir, 'protac_splitter'))

from protac_splitter.evaluation import (
    check_reassembly,
    split_prediction,
)
from protac_splitter.chemoinformatics import (
    dummy2query,
    canonize,
)

In [87]:
protac_examples = [
        [
            'N#Cc1ccc(O[C@H]2CC[C@H](NC(=O)c3ccc(N4CCN(CCCCCNc5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)cc3)CC2)cc1Cl',
            '[*:1]N[C@@H]2CC[C@@H](Oc1ccc(C#N)c(Cl)c1)CC2.[*:2]CCCCCN2CCN(c1ccc(C([*:1])=O)cc1)CC2.[*:2]Nc3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
        [
            'CN(c1ccc(C#N)c(Cl)c1)[C@H]1CC[C@H](NC(=O)c2ccc(N3CC(CN4CCN(c5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)C3)cc2)CC1',
            '[*:1]N[C@@H]2CC[C@@H](N(C)c1ccc(C#N)c(Cl)c1)CC2.[*:1]C(=O)c3ccc(N2CC(CN1CCN([*:2])CC1)C2)cc3.[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
        [
            'CN1C(=O)CCc2cc3cc(c21)OCCOCC1CN(C(=O)CCC(=O)NCCCOCCOCCOc2cccc4c2C(=O)N(C2CCC(=O)NC2=O)C4=O)CCN1c1ncc(Cl)c(n1)N3',
            '[*:1]N5CCN4c1ncc(Cl)c(n1)Nc3cc2CCC(=O)N(C)c2c(c3)OCCOCC4C5.[*:2]OCCOCCOCCCNC(=O)CCC([*:1])=O.[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'N#CC1(CNc2cccc(-c3cc(N[C@H]4CC[C@H](NCC(=O)NCCOCCOCCOCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ncc3Cl)n2)CCOCC1',
            '[*:1]C(=O)CN[C@@H]4CC[C@@H](Nc3cc(c2cccc(NCC1(C#N)CCOCC1)n2)c(Cl)cn3)CC4.[*:2]NCCOCCOCCOCCN[*:1].[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'O=C1CCC(N2C(=O)c3ccc(OCCOCCOCCOCCN4CCN(Cc5ccc6nc(NC(=O)c7cccc(C(F)(F)F)c7)n([C@H]7CC[C@@H](CO)CC7)c6c5)CC4)cc3C2=O)C(=O)N1',
            '[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3.[*:1]CN1CCN(CCOCCOCCOCCO[*:2])CC1.[*:1]c4ccc3nc(NC(=O)c1cccc(C(F)(F)F)c1)n([C@@H]2CC[C@H](CO)CC2)c3c4',
        ],
        [
            'N#Cc1ccc(O[C@H]2CC[C@H](NC(=O)c3ccc(N4CCN(CCCCNc5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)cc3)CC2)cc1Cl',
            '[*:1]N[C@@H]2CC[C@@H](Oc1ccc(C#N)c(Cl)c1)CC2.[*:2]NCCCCN2CCN(c1ccc(C([*:1])=O)cc1)CC2.[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
        [
            'Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)COCCOCCOCCNC(=O)CCC(=O)N2CCN([C@H]3CC[C@@H](Nc4ncnn5ccc(C(C)C)c45)CC3)CC2)C(C)(C)C)cc1',
            '[*:2]N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)NCc3ccc(c2scnc2C)cc3)C(C)(C)C.[*:1]C(=O)CCC(=O)NCCOCCOCCOCC([*:2])=O.[*:1]N4CCN([C@@H]3CC[C@H](Nc1ncnn2ccc(C(C)C)c12)CC3)CC4',
        ],
        [
            'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CNC(=O)c2cccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)c2)C(C)(C)C)cc1',
            '[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc3ccc(c2scnc2C)cc3)C(C)(C)C.[*:1]NCC([*:2])=O.[*:1]C(=O)c4cccc(c3ccc(N1CCN(C)CC1)c(NC(=O)c2c[nH]c(=O)cc2C(F)(F)F)c3)c4',
        ],
        [
            'CN1C(=O)CCc2cc3cc(c21)OCCOC[C@H]1CN(C(=O)CCC(=O)NCCCOCCOCCOc2cccc4c2C(=O)N(C2CCC(=O)NC2=O)C4=O)CCN1c1ncc(Cl)c(n1)N3',
            '[*:1]N5CCN4c1ncc(Cl)c(n1)Nc3cc2CCC(=O)N(C)c2c(c3)OCCOC[C@H]4C5.[*:2]OCCOCCOCCCNC(=O)CCC([*:1])=O.[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'CC(C)Nc1cc(-n2ccc3cc(C#N)cnc32)ncc1C(=O)N[C@H]1CC[C@H](C(=O)NCCOCCOCCOCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1',
            '[*:1]C(=O)[C@@H]4CC[C@@H](NC(=O)c3cnc(n2ccc1cc(C#N)cnc12)cc3NC(C)C)CC4.[*:1]NCCOCCOCCOCCNC(=O)CO[*:2].[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],

        [
            'CC(C)Nc1cc(-n2ccc3cc(C#N)cnc32)ncc1C(=O)N[C@H]1CC[C@H](C(=O)NCCOCCOCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1',
            '[*:1]C(=O)[C@@H]4CC[C@@H](NC(=O)c3cnc(n2ccc1cc(C#N)cnc12)cc3NC(C)C)CC4.[*:1]NCCOCCOCCNC(=O)CO[*:2].[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CCCNC(=O)CCC(=O)N2CCN([C@H]3CC[C@@H](Nc4ncnn5ccc(C(C)C)c45)CC3)CC2)C(C)(C)C)cc1',
            '[*:2]N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)NCc3ccc(c2scnc2C)cc3)C(C)(C)C.[*:2]C(=O)CCCNC(=O)CCC([*:1])=O.[*:1]N4CCN([C@@H]3CC[C@H](Nc1ncnn2ccc(C(C)C)c12)CC3)CC4',
        ],
        [
            'CC(C)Nc1cc(-n2ccc3cc(C#N)cnc32)ncc1C(=O)N[C@H]1CC[C@H](C(=O)NCCOCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1',
            '[*:1]C(=O)c3cnc(n2ccc1cc(C#N)cnc12)cc3NC(C)C.[*:1]N[C@@H]1CC[C@@H](C(=O)NCCOCCNC(=O)CO[*:2])CC1.[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'O=C1CCC(N2C(=O)c3ccc(OCCOCCOCCN4CCN(Cc5ccc6nc(NC(=O)c7cccc(C(F)(F)F)c7)n([C@H]7CC[C@@H](CO)CC7)c6c5)CC4)cc3C2=O)C(=O)N1',
            '[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3.[*:1]CCOCCOCCO[*:2].[*:1]N5CCN(Cc4ccc3nc(NC(=O)c1cccc(C(F)(F)F)c1)n([C@@H]2CC[C@H](CO)CC2)c3c4)CC5',
        ],
        [
            'N#Cc1ccc(O[C@H]2CC[C@H](NC(=O)c3ccc(NCCCNc4ccc5c(c4)C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)CC2)cc1Cl',
            '[*:1]N[C@@H]2CC[C@@H](Oc1ccc(C#N)c(Cl)c1)CC2.[*:2]NCCCNc1ccc(C([*:1])=O)cc1.[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
        [
            'CC(C)Nc1cc(-n2ccc3cc(C#N)cnc32)ncc1C(=O)N[C@H]1CC[C@H](C(=O)NCCCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1',
            '[*:1]C(=O)c3cnc(n2ccc1cc(C#N)cnc12)cc3NC(C)C.[*:1]N[C@@H]1CC[C@@H](C(=O)NCCCCNC(=O)CO[*:2])CC1.[*:2]c2cccc3c(=O)n(C1CCC(=O)NC1=O)c(=O)c23',
        ],
        [
            'Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CN2CCN(CCN3CCC(O[C@H]4C[C@H](Oc5ccc6c(c5)Sc5cc([N+](=O)[O-])ccc5N6)C4)CC3)CC2)C(C)(C)C)cc1',
            '[*:2]N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)N[C@@H](C)c3ccc(c2scnc2C)cc3)C(C)(C)C.[*:1]O[C@@H]3C[C@@H](OC2CCN(CCN1CCN(CC([*:2])=O)CC1)CC2)C3.[*:1]c3ccc2[nH]c1ccc(N(=O)=O)cc1sc2c3',
        ],
        [
            'N#Cc1ccc(O[C@H]2CC[C@H](NC(=O)c3ccc(NCCCCCCCCCNc4ccc5c(c4)C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)CC2)cc1Cl',
            '[*:1]N[C@@H]2CC[C@@H](Oc1ccc(C#N)c(Cl)c1)CC2.[*:2]NCCCCCCCCCNc1ccc(C([*:1])=O)cc1.[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
        [
            'N#Cc1ccc(O[C@H]2CC[C@H](NC(=O)c3ccc(NCCCCCCCCCNc4ccc5c(c4)C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)CC2)cc1Cl',
            '[*:1]N[C@@H]2CC[C@@H](Oc1ccc(C#N)c(Cl)c1)CC2.[*:2]CCC[*:1].[*:2]c3ccc2c(=O)n(C1CCC(=O)NC1=O)c(=O)c2c3',
        ],
    ]

In [88]:
# Set logging level so that we can debug the code
logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [ ]:
def remove_attach_atom(mol: Chem.Mol, attach_id: int) -> Chem.Mol:
    """ Removes the atom with the specified attachment id from the molecule.

    Args:
        mol: The molecule.
        attach_id: The attachment id of the atom to remove.

    Returns:
        The molecule with the atom removed.
    """
    atoms_to_remove = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 0:  # Dummy atom
            map_num = atom.GetAtomMapNum()
            if map_num == attach_id:  # Targeting only [*:attach_id]
                atoms_to_remove.append(atom.GetIdx())

    # Remove atoms using an EditableMol
    editable_mol = Chem.EditableMol(mol)
    for idx in sorted(atoms_to_remove, reverse=True):  # Remove from highest index to avoid shifting
        editable_mol.RemoveAtom(idx)

    # Convert back to a molecule
    new_mol = editable_mol.GetMol()
    Chem.SanitizeMol(new_mol)
    return new_mol


def fix_prediction(
        protac_smiles: str,
        pred_smiles: str,
        poi_attachment_id: int = 1,
        e3_attachment_id: int = 2,
        remove_stereochemistry: bool = False,
) -> Optional[Dict[str, str]]:
    """ Fixes a prediction by replacing the substructure that does not match the PROTAC with the rest of the PROTAC.
    
    Args:
        protac_smiles: The SMILES of the PROTAC.
        pred_smiles: The SMILES of the prediction.
        poi_attachment_id: The attachment point id of the POI. Default is 1.
        e3_attachment_id: The attachment point id of the E3 ligase. Default is 2.

    Returns:
        A dictionary (with keys: 'e3', 'linker', 'poi') containing the fixed substructures, or None if the prediction is invalid.
    """
    
    substructs = split_prediction(pred_smiles)

    # If there are at least two None values, there's nothing we can do to fix it
    if sum(v is None for v in substructs.values()) >= 2:
        print(f'Unable to continue, more than two substructures are not valid for given input: "{pred_smiles}"')
        return None

    # Get molecules of PROTAC and substructures
    protac_mol = Chem.MolFromSmiles(protac_smiles)
    substructs = {k: {'smiles': v, 'mol': Chem.MolFromSmiles(v) if v is not None else v} for k, v in substructs.items()}

    # Check if removing stereochemistry results in a valid prediction
    if remove_stereochemistry:
        Chem.RemoveStereochemistry(protac_mol)
        protac_smiles = Chem.MolToSmiles(protac_mol, canonical=True)
        for k, v in substructs.items():
            if v['mol'] is not None:
                Chem.RemoveStereochemistry(v['mol'])
                substructs[k]['smiles'] = Chem.MolToSmiles(v['mol'], canonical=True)
    
    if all(v['mol'] is not None for v in substructs.values()):
        if check_reassembly(
            protac_smiles,
            '.'.join([v['smiles'] for v in substructs.values()]),
        ):
            logging.info(f'Input works when removing stereochemistry. SMILES: "{pred_smiles}"')
            return f"{substructs['e3']['smiles']}.{substructs['linker']['smiles']}.{substructs['poi']['smiles']}"

    # Check if any of the substructures is NOT a substructure of the PROTAC, if
    # so, we mark it as the wrong substructure to fix.
    num_matches = 0
    wrong_substruct = None
    for sub in ['poi', 'linker', 'e3']:
        if substructs[sub]['mol'] is None:
            substructs[sub]['match'] = False
            wrong_substruct = sub
        elif protac_mol.HasSubstructMatch(dummy2query(substructs[sub]['mol'])):
            substructs[sub]['match'] = True
            num_matches += 1
        else:
            substructs[sub]['match'] = False
            wrong_substruct = sub

    if num_matches < 2:
        print(f'Prediction does not contain at least two matching substructures of the PROTAC. Num matches: {num_matches}. Prediction SMILES: "{pred_smiles}"')
        return None

    # If the wrong substructure is still matching in the PROTAC, we need to a
    # more complex approach to fix the prediction (see below).
    def remove_substructure(mol, substructure, attachment_id, replaceDummies=False):
        if mol is None or substructure is None:
            return None
        smaller_mol = Chem.ReplaceCore(
            mol,
            substructure,
            labelByIndex=False,
            replaceDummies=replaceDummies,
        )
        if smaller_mol is None:
            print(f'Failed to remove substructure from prediction SMILES: "{pred_smiles}"')
            return None
        smaller_smiles = Chem.MolToSmiles(smaller_mol, canonical=True)
        smaller_smiles = smaller_smiles.replace('[1*]', f'[*:{attachment_id}]')
        smaller_smiles = smaller_smiles.replace('[2*]', f'[*:{attachment_id}]')
        smaller_mol = canonize(Chem.MolFromSmiles(smaller_smiles))
        return smaller_mol

    # If we still have 3 matches: for each substructure, we progressively remove
    # the other substructures, then we check if the resulting molecule is valid
    # and has only one fragment.
    if num_matches == 3:
        wrong_substruct = None
        for sub in ['poi', 'linker', 'e3']:
            removed_mol = Chem.MolFromSmiles(protac_smiles)

            # Put the current substructure at the end of the list [poi, e3, linker]
            sub_names = ['poi', 'e3', 'linker']
            sub_names.remove(sub)
            sub_names.append(sub)
            # The linker often matches in many parts of the PROTAC, so we remove
            # it when checking the E3 and POI substructures.
            if sub != 'linker':
                sub_names.remove('linker')

            for s in sub_names:
                attachment_id = poi_attachment_id if s == 'poi' else e3_attachment_id
                removed_mol = remove_substructure(
                    removed_mol,
                    dummy2query(substructs[s]['mol']),
                    attachment_id=attachment_id,
                )

            # Check if resulting molecule is None, if so, it is the wrong one
            if removed_mol is None:
                substructs[sub]['match'] = False
                wrong_substruct = sub
                num_matches -= 1
                break

            # Count the number of fragments in the removed molecule
            num_fragments = Chem.GetMolFrags(removed_mol, asMols=True, sanitizeFrags=False)
            if len(num_fragments) > 1:
                substructs[sub]['match'] = False
                wrong_substruct = sub
                num_matches -= 1
                break

    if num_matches == 3:
        print(f'Prediction already contains all matching substructures of the PROTAC. Prediction SMILES: "{pred_smiles}"')
        return None

    # Get the order in which to remove the substructures and get the final one
    # as the fixed molecule.
    if wrong_substruct == 'linker':
        poi_atoms = substructs['poi']['mol'].GetNumAtoms()
        e3_atoms = substructs['e3']['mol'].GetNumAtoms()
        order = ['poi', 'e3'] if poi_atoms > e3_atoms else ['e3', 'poi']
    else:
        if wrong_substruct == 'poi':
            order = ['e3', 'linker']
        else:
            order = ['poi', 'linker']

    print(f'Wrong substructure: {wrong_substruct.upper()}. Order: {order}')

    fixed_mol = protac_mol
    for sub in order:
        print(f'Removing substructure {sub.upper()} from PROTAC.')

        if 'linker' not in order:
            fixed_attach_id = poi_attachment_id if sub == 'poi' else e3_attachment_id
        else:
            fixed_attach_id = poi_attachment_id if 'e3' in order else e3_attachment_id

        if sub == 'linker':
            attach_id = poi_attachment_id if wrong_substruct == 'poi' else e3_attachment_id
            fixed_attach_id = poi_attachment_id if wrong_substruct == 'poi' else e3_attachment_id
            query_mol = remove_attach_atom(substructs[sub]['mol'], attach_id)
            replaceDummies = True
        else:
            query_mol = dummy2query(substructs[sub]['mol'])
            replaceDummies = False

        display(Draw.MolToImage(fixed_mol, legend=f"Starting molecule", size=(800, 300)))
        display(Draw.MolToImage(query_mol, legend=f"Molecule {sub.upper()} to remove", size=(800, 300)))

        fixed_mol_tmp = remove_substructure(
            fixed_mol,
            query_mol,
            attachment_id=fixed_attach_id,
            replaceDummies=replaceDummies,
        )
        if fixed_mol_tmp is None:
            print(f'Failed to replace substructure "{sub}" in prediction SMILES: "{pred_smiles}"')
            continue

        fixed_mol = fixed_mol_tmp

        # If there are multiple fragments, keep the biggest one
        fragments = Chem.GetMolFrags(fixed_mol, asMols=True)
        if len(fragments) > 1:
            print(f'Fixed molecule contains more than one fragment. Keeping the biggest one.')
            max_frag = max(fragments, key=lambda x: x.GetNumAtoms())
            fixed_mol = max_frag

    # Get the SMILES of the fixed molecule
    fixed_smiles = Chem.MolToSmiles(canonize(fixed_mol), canonical=True)
    substructs[wrong_substruct]['smiles'] = fixed_smiles

    display(Draw.MolToImage(fixed_mol, legend=f"{wrong_substruct.upper()} fixed molecule: {fixed_smiles}", size=(800, 300)))

    # Concatenate the substructures check if the re-assembly is correct
    fixed_pred_smiles = f"{substructs['e3']['smiles']}.{substructs['linker']['smiles']}.{substructs['poi']['smiles']}"

    if not check_reassembly(
        protac_smiles,
        fixed_pred_smiles,
    ):
        # print(f"Failed to fix prediction, re-assembly check failed. Substructures: {fixed_pred_smiles}")
        # return fixed_pred_smiles
        return None

    return fixed_pred_smiles



def make_sub_none(substructs: Dict[str, str]) -> Dict[str, str]:
    """ Makes a random substructure None in the substructs dictionary. """
    sub = random.choice(list(substructs.keys()))
    substructs[sub] = 'wrong_substructure'
    return substructs

def add_atom(substructs: Dict[str, str]) -> Dict[str, str]:
    """ Adds a random atom to a random substructure in the substructs dictionary. """
    subs = ['e3', 'linker', 'poi']
    random.shuffle(subs)
    for sub in subs:
        if 'CC' in substructs[sub]:
            print(f'Removing one atom from substructure: {sub.upper()}')
            substructs[sub] = substructs[sub].replace('CC', 'C', 1)
            break
    return substructs

def add_extra_atoms(substructs: Dict[str, str]) -> Dict[str, str]:
    subs = ['e3', 'linker', 'poi']
    random.shuffle(subs)
    for sub in subs:
        if 'CC' in substructs[sub]:
            num_errors = random.choice([1, 2])
            error_atoms = 'CC' + 'C' * num_errors
            substructs[sub] = substructs[sub].replace('CC', error_atoms, 1)
            print(f'Adding N.{num_errors} atom to substructure: {sub.upper()}')
            break
    return substructs

def alter_atom(substructs: Dict[str, str]) -> Dict[str, str]:
    subs = ['e3', 'linker', 'poi']
    random.shuffle(subs)
    for sub in subs:
        if 'N' in substructs[sub]:
            print(f'Altering one atom from substructure: {sub.upper()}')
            # Randomly select a "N" atom and replace it with "C"
            n_atoms = [i for i, c in enumerate(substructs[sub]) if c == 'N']
            if len(n_atoms) == 0:
                continue
            n_atom = random.choice(n_atoms)
            substructs[sub] = substructs[sub][:n_atom] + 'C' + substructs[sub][n_atom + 1:]
            break
    return substructs

for i in range(5):
    random.seed(42 + i)

    error_functions = [
        make_sub_none,
        add_atom,
        add_extra_atoms,
        alter_atom,
    ]
    for error_function in error_functions:
        print('-' * 100)
        print(f"Testing error function: {error_function.__name__}")
        print('-' * 100)
        for protac_smiles, pred_smiles in protac_examples:
            protac_smiles = canonize(protac_smiles)
            pred_smiles = canonize(pred_smiles)
            
            protac_mol = Chem.MolFromSmiles(pred_smiles)
            protac_mol = canonize(Chem.molzip(protac_mol))
            protac_smiles = Chem.MolToSmiles(protac_mol, canonical=True)

            substructs = split_prediction(pred_smiles)
            label_smiles = f"{substructs['e3']}.{substructs['linker']}.{substructs['poi']}"

            substructs = error_function(substructs)
            pred_smiles = '.'.join([substructs[s] for s in ['e3', 'linker', 'poi']])
            
            print(f'PROTAC: {protac_smiles}')
            print(f'Label:  {label_smiles}')
            print(f'Pred:   {pred_smiles}')

            fixed_smiles = fix_prediction(protac_smiles, pred_smiles)

            print(f'PROTAC: {protac_smiles}')
            print(f'Pred:   {pred_smiles}')
            print(f'Label:  {label_smiles}')
            print(f'Fixed:  {fixed_smiles}')

            if fixed_smiles is None:
                display(Chem.MolFromSmiles(protac_smiles))
            assert fixed_smiles is not None, f'Failed to fix prediction for "{pred_smiles}"'
            assert fixed_smiles == label_smiles, f'Fixed prediction is not the same as the original prediction for "{pred_smiles}"'
            print('-' * 80)